While building the model (Task 2) proves how well it works, Model Explainability (Task 3) proves why it works. This is what you will actually present to business stakeholders.

The Python Code for Explainability
This code assumes you are continuing from Task 2 and have your trained model (e.g., best_xgb), your test features (X_test), your true labels (y_test), and your model's predictions (y_pred).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap

# NOTE: If running in a Jupyter Notebook, initialize JS visualization for SHAP
shap.initjs()

# ==========================================
# 1. Feature Importance Baseline (XGBoost Built-in)
# ==========================================
print("Generating Built-in Feature Importance...")

# Extract feature importances and feature names
importances = best_xgb.feature_importances_
features = X_test.columns

# Create a DataFrame and sort
importance_df = pd.DataFrame({'Feature': features, 'Importance': importances})
importance_df = importance_df.sort_values(by='Importance', ascending=False).head(10)

# Plot Built-in Importance
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=importance_df, palette='viridis')
plt.title('Top 10 Features: XGBoost Built-in Importance')
plt.xlabel('Relative Importance (Gain)')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

# ==========================================
# 2. SHAP Analysis: Global Explainability
# ==========================================
print("Calculating SHAP values (this may take a moment)...")

# Initialize the TreeExplainer
explainer = shap.TreeExplainer(best_xgb)

# Calculate SHAP values for the test set
# (If X_test is very large, consider using a random sample like X_test.sample(2000) to speed this up)
shap_values = explainer.shap_values(X_test)

# Generate SHAP Summary Plot
plt.figure()
plt.title('SHAP Summary Plot (Global Feature Importance)')
shap.summary_plot(shap_values, X_test, show=False)
plt.tight_layout()
plt.show()

# ==========================================
# 3. SHAP Analysis: Local Explainability (Force Plots)
# ==========================================
# First, identify the indices for TP, FP, and FN in our test set
y_test_array = y_test.values
y_pred_array = y_pred # Assuming y_pred from Task 2 is a numpy array

tp_indices = np.where((y_test_array == 1) & (y_pred_array == 1))[0]
fp_indices = np.where((y_test_array == 0) & (y_pred_array == 1))[0]
fn_indices = np.where((y_test_array == 1) & (y_pred_array == 0))[0]

# Helper function to plot and save/show force plots
def generate_force_plot(idx, title_label):
    print(f"Generating Force Plot for: {title_label} (Index: {idx})")
    # Generate the plot
    p = shap.force_plot(
        explainer.expected_value, 
        shap_values[idx, :], 
        X_test.iloc[idx, :], 
        matplotlib=True, # Set to False if using Jupyter and prefer interactive JS plots
        show=False
    )
    plt.title(f'SHAP Force Plot: {title_label}')
    plt.show()

# Ensure we have at least one of each before plotting
if len(tp_indices) > 0:
    generate_force_plot(tp_indices[0], "True Positive (Correctly Flagged Fraud)")
if len(fp_indices) > 0:
    generate_force_plot(fp_indices[0], "False Positive (Legitimate but Flagged)")
if len(fn_indices) > 0:
    generate_force_plot(fn_indices[0], "False Negative (Missed Fraud)")